In [3]:
import osmnx as ox

tags = {
    "tourism": True,
    "historic": True,
    "leisure": ["park", "garden"],
    "amenity": ["restaurant", "cafe", "place_of_worship"],
}

center = (25.31, 83.00)  # near the ghats, approximate
gdf = ox.features_from_point(center, tags, dist=7000)

print(len(gdf), gdf["name"].notna().sum(), gdf["opening_hours"].notna().sum() if "opening_hours" in gdf else 0)

942 671 44


In [4]:
gdf.to_pickle("../data/raw/varanasi_osm_raw.pkl")


In [5]:
for col in ["tourism", "historic", "leisure", "amenity"]:
    if col in gdf:
        print(gdf[col].value_counts().head(12), "\n")

tourism
attraction     111
hotel          106
hostel          94
guest_house     64
artwork         12
apartment        8
information      6
museum           5
viewpoint        3
gallery          2
picnic_site      1
motel            1
Name: count, dtype: int64 

historic
memorial               13
city_gate              10
monument                2
castle                  1
archaeological_site     1
building                1
Name: count, dtype: int64 

leisure
park      90
garden    29
Name: count, dtype: int64 

amenity
restaurant          171
place_of_worship    144
cafe                 69
marketplace           1
Name: count, dtype: int64 



In [6]:
acc = ["hotel", "hostel", "guest_house", "apartment", "motel"]
named = gdf[gdf["name"].notna() & ~gdf["tourism"].isin(acc)].copy()
print("named, non-accommodation:", len(named))

ghat = gdf[gdf["name"].fillna("").str.contains("ghat", case=False)]
print("names containing 'ghat':", len(ghat))
print(ghat["name"].head(20).tolist())

print("with wikidata:", gdf["wikidata"].notna().sum() if "wikidata" in gdf else 0)

named, non-accommodation: 455
names containing 'ghat': 84
['Hanuman Ghat', 'Ahilyabai Ghat', 'Raja Ghat', 'Ranamahal Ghat', 'Manasarowa Ghat', 'Manmandir Ghat', 'Kedar Ghat', 'Badhaini Ghat', 'Ganga Mahal Ghat', 'Lali Ghat', 'Babua Pandey Ghat', 'Munshi Ghat', 'Karnataka Ghat', 'Narad Ghat', 'Chousatti Ghat', 'Ganesh Ghat', 'Dr. Rajendra Prasad Ghat', 'Tulsi Ghat', 'Sankatha Ghat', 'Anandamayee Ghat']
with wikidata: 9


In [7]:
import os

acc = ["hotel", "hostel", "guest_house", "apartment", "motel"]
df = gdf[gdf["name"].notna() & ~gdf["tourism"].isin(acc)].copy()

def categorize(r):
    n = str(r["name"]).lower()
    if "ghat" in n: return "ghat"
    if r.get("amenity") == "place_of_worship": return "worship"
    if r.get("amenity") == "restaurant": return "restaurant"
    if r.get("amenity") == "cafe": return "cafe"
    if r.get("leisure") in ("park", "garden"): return "park_garden"
    if r.get("tourism") in ("museum", "gallery"): return "museum"
    if pd.notna(r.get("historic")): return "historic"
    if r.get("tourism") in ("attraction", "viewpoint", "artwork"): return "attraction"
    return "other"

import pandas as pd
df["category"] = df.apply(categorize, axis=1)
pts = df.geometry.representative_point()
df["lat"], df["lon"] = pts.y, pts.x

out = df.reset_index()[["name", "category", "lat", "lon", "opening_hours"]].drop_duplicates(["name", "category"])
os.makedirs("../data/processed", exist_ok=True)
out.to_csv("../data/processed/varanasi_places_clean.csv", index=False)

print(out["category"].value_counts())
print(out["lat"].min(), out["lat"].max(), out["lon"].min(), out["lon"].max())
for k in ["dashashwamedh", "assi", "manikarnika", "panchganga"]:
    print(k, out["name"].str.contains(k, case=False).sum())

category
restaurant     131
worship         88
ghat            79
park_garden     57
cafe            49
historic        16
attraction      11
museum           7
other            3
Name: count, dtype: int64
25.2538619 25.371897 82.9329633 83.0442908
dashashwamedh 0
assi 4
manikarnika 1
panchganga 1


In [8]:
import pandas as pd
out = pd.read_csv("../data/processed/varanasi_places_clean.csv")
for k in ["dasha", "vishwanath", "sankat", "durga", "manikarnika", "bharat mata", "sarnath", "ramnagar"]:
    print(k, out.loc[out["name"].str.contains(k, case=False), ["name", "category"]].values.tolist()[:4])

dasha [['Dashashvamedh Ghat', 'ghat']]
vishwanath [['vishwanath ashram', 'worship'], ['Vishwanath Temple', 'worship'], ['Kashi Vishwanath Temple', 'worship'], ['Kashi Vishwanath Dham, varanasi', 'worship']]
sankat [['Sankatha Ghat', 'ghat']]
durga [['Durga Temple', 'worship'], ['Maa Durga  Mandir', 'worship'], ['Durgakund', 'worship'], ['Durga Mata Mandir', 'worship']]
manikarnika [['Manikarnika Ghat', 'ghat']]
bharat mata []
sarnath []
ramnagar [['Ramnagar Fort', 'museum'], ['Ramnagar fort', 'attraction'], ['Rambagh Ramnagar', 'park_garden']]


In [9]:
import re, difflib
from itertools import combinations
import pandas as pd

out = pd.read_csv("../data/processed/varanasi_places_clean.csv")
norm = lambda s: re.sub(r"[^a-z ]", "", str(s).lower()).replace("mandir", "temple").strip()
out["key"] = out["name"].map(norm)
out = out.drop_duplicates("key").reset_index(drop=True)

drop = set()
for i, j in combinations(range(len(out)), 2):
    a, b = out.loc[i], out.loc[j]
    if abs(a.lat - b.lat) < 0.001 and abs(a.lon - b.lon) < 0.001:
        if a.key in b.key or b.key in a.key or difflib.SequenceMatcher(None, a.key, b.key).ratio() > 0.6:
            print(a["name"], "|", b["name"])
            drop.add(j)

out = out.drop(index=drop).drop(columns="key").reset_index(drop=True)
out.to_csv("../data/processed/varanasi_places_dedup.csv", index=False)
print(len(out))

quota = {"ghat": 8, "worship": 6, "restaurant": 8, "cafe": 5,
         "park_garden": 4, "historic": 3, "attraction": 3, "museum": 3}
parts = [out[out["category"] == c].sample(min(n, (out["category"] == c).sum()), random_state=42)
         for c, n in quota.items()]
survey = pd.concat(parts).reset_index(drop=True)
survey.to_csv("../data/processed/survey_candidates.csv", index=False)
print(survey[["name", "category"]].to_string())

Hanuman Ghat | Karnataka Ghat
Hanuman Ghat | Shivala Ghat
Ahilyabai Ghat | Darbhanga Ghat
Ahilyabai Ghat | Prayag Ghat
Raja Ghat | Narad Ghat
Ranamahal Ghat | Darbhanga Ghat
Manasarowa Ghat | Narad Ghat
Manmandir Ghat | Meer Ghat
Kedar Ghat | Lali Ghat
Badhaini Ghat | janki ghat
Ganga Mahal Ghat | Sankatha Ghat
Karnataka Ghat | Gularia Ghat
Sankatha Ghat | Scindia Ghat
Sankatha Ghat | Agrodha Ghat
Karki's Restaurant | Angeethee Resturant
Karki's Restaurant | Carvaan Cafe & Restaurant
Gaay Ghat | Badrinarayan Ghat
Jatar Ghat | Gwaljor Ghat
Jatar Ghat | Ram Ghat
Lal Ghat | Shitala Ghat
Lalita Ghat | Jalasayi Ghat
Mahanirvani Ghat | Niranjani Ghat
Mahanirvani Ghat | Shivala Ghat
Mehfa Ghat | Ram Ghat
Mehfa Ghat | Agrodha Ghat
Niranjani Ghat | Shivala Ghat
Naya Ghat | Prahlad Ghat
Panchkot Ghat | Prabhu Ghat
Panchkot Ghat | Jain Ghat
Gwaljor Ghat | Balaji Ghat
Ram Ghat | Balaji Ghat
Trilochan Ghat | Gola Ghat
Adi Keshava Ghat | Adi Keshav Temple
Shitala Ghat | Brahma Ghat
Shiva Restaurant 

In [10]:
import re, difflib
import pandas as pd

out = pd.read_csv("../data/processed/varanasi_places_clean.csv")

junk = r"canteen|canteeen|hostel|\bblock\b|gmcha|parking|\bvt\b|boto|\bgate\b|statue|campus|\bhut\b"
generic = {"mandir", "park", "local park", "public garden", "private garden", "garden", "temple", "restaurant", "cafe"}
out = out[~out["name"].str.lower().str.contains(junk, regex=True)]
out = out[~out["name"].str.lower().str.strip().isin(generic) & (out["name"].str.len() >= 4)].reset_index(drop=True)

stop = {"ghat", "temple", "mandir", "restaurant", "resturant", "cafe", "garden", "park", "shri", "shree", "the"}
def core(s):
    return " ".join(w for w in re.sub(r"[^a-z ]", "", str(s).lower()).split() if w not in stop)
out["key"] = out["name"].map(core)

drop = set()
for c, g in out.groupby("category"):
    idx = list(g.index)
    for a in range(len(idx)):
        for b in range(a + 1, len(idx)):
            i, j = idx[a], idx[b]
            if i in drop or j in drop or not out.key[i] or not out.key[j]:
                continue
            close = abs(out.lat[i] - out.lat[j]) < 0.001 and abs(out.lon[i] - out.lon[j]) < 0.001
            if close and difflib.SequenceMatcher(None, out.key[i], out.key[j]).ratio() > 0.85:
                print(out.name[i], "|", out.name[j])
                drop.add(j)

out = out.drop(index=drop).drop(columns="key").reset_index(drop=True)
out.to_csv("../data/processed/varanasi_places_dedup.csv", index=False)
print(len(out))
print(out["category"].value_counts())

Madhuban Park | Madhuban Garden
Sagar Ratna | sagar ratna
AM 2 PM Restaurant | AM2PM RESTAURANT
388
category
restaurant     124
worship         86
ghat            79
cafe            40
park_garden     35
attraction       9
museum           7
historic         6
other            2
Name: count, dtype: int64


In [11]:
import pandas as pd
out = pd.read_csv("../data/processed/varanasi_places_dedup.csv")
for k in ["assi", "harish", "sankat", "tulsi", "man singh", "bharat kala", "durga", "blue lassi", "kachori"]:
    print(k, out.loc[out["name"].str.contains(k, case=False), ["name", "category"]].values.tolist()[:4])

assi [['Blue Lassi Shop', 'cafe'], ['Bowl of compassion', 'cafe'], ['Classic Soya Chaap Zone', 'restaurant'], ['Assi Ghat Mandir', 'ghat']]
harish [['Harishchandra Ghat', 'ghat']]
sankat [['Sankatha Ghat', 'ghat']]
tulsi [['Tulsi Ghat', 'ghat'], ['Tulsi Manas Temple', 'worship']]
man singh [['Man Singh Astronomical Observatory', 'museum'], ['Man Singh Palace', 'museum']]
bharat kala [['Bharat Kala Bhavan', 'museum']]
durga [['Durga Temple', 'worship'], ['Maa Durga  Mandir', 'worship'], ['Durgakund', 'worship'], ['Durga Mata Mandir', 'worship']]
blue lassi [['Blue Lassi Shop', 'cafe']]
kachori []


In [12]:
import re
pat = r"\bassi\b|hanuman|dashashvamedh|dasaswamedh|manikarnika|kashi vishwanath|ramnagar fort|bharat mata|संकट|काशी"
hits = out[out["name"].str.contains(pat, case=False, regex=True)]
print(hits[["name", "category"]].to_string())

                                       name    category
0                    संकट मोचन हनुमान मंदिर     worship
2                              Hanuman Ghat        ghat
26                            Ramnagar Fort      museum
41                         Manikarnika Ghat        ghat
67                        Hanumangadhi Ghat        ghat
139                        Dasaswamedh Ghat        ghat
143  Shri Senapathi Banakatti Hanuman Tmple     worship
149                           Ramnagar fort  attraction
150                          Hanuman Temple     worship
158               Muchh wale Hanuman mandir     worship
209                      Dashashvamedh Ghat        ghat
224                          hanuman mandir     worship
259                          Hanuman Mandir     worship
375                 Kashi Vishwanath Temple     worship
376         Kashi Vishwanath Dham, varanasi     worship
377                shri bade hanuman mandir     worship
380                     Lord Hanuman Temple     

In [13]:
fix = {
    "Shri Senapathi Banakatti Hanuman Tmple": "Shri Senapathi Banakatti Hanuman Temple",
    "संकट मोचन हनुमान मंदिर": "Sankat Mochan Hanuman Temple",
}
out["name"] = out["name"].replace(fix)

remove = ["Dasaswamedh Ghat", "Ramnagar fort", "Kashi Vishwanath Dham, varanasi"]
out = out[~out["name"].isin(remove)].reset_index(drop=True)
out.to_csv("../data/processed/varanasi_places_final.csv", index=False)

famous = ["Dashashvamedh Ghat", "Manikarnika Ghat", "Harishchandra Ghat", "Tulsi Ghat",
          "Kashi Vishwanath Temple", "Sankat Mochan Hanuman Temple", "Tulsi Manas Temple",
          "Durga Temple", "Ramnagar Fort", "Man Singh Palace", "Bharat Kala Bhavan", "Blue Lassi Shop"]
print(len(out))
print(out[out["name"].isin(famous)][["name", "category", "lat", "lon"]].to_string())
print("missing:", [f for f in famous if f not in set(out["name"])])

385
                             name category        lat        lon
0    Sankat Mochan Hanuman Temple  worship  25.282139  82.999981
24                     Tulsi Ghat     ghat  25.290240  83.006343
26                  Ramnagar Fort   museum  25.270584  83.024878
36             Harishchandra Ghat     ghat  25.298315  83.007190
41               Manikarnika Ghat     ghat  25.310768  83.014021
79                Blue Lassi Shop     cafe  25.312373  83.011354
96                   Durga Temple  worship  25.288607  82.999274
207            Dashashvamedh Ghat     ghat  25.306930  83.010654
320              Man Singh Palace   museum  25.307753  83.010830
339            Bharat Kala Bhavan   museum  25.271432  82.995792
346            Tulsi Manas Temple  worship  25.287147  83.000400
373       Kashi Vishwanath Temple  worship  25.310822  83.010668
missing: []


In [14]:
famous = ["Dashashvamedh Ghat", "Manikarnika Ghat", "Harishchandra Ghat", "Tulsi Ghat",
          "Kashi Vishwanath Temple", "Sankat Mochan Hanuman Temple", "Tulsi Manas Temple",
          "Durga Temple", "Ramnagar Fort", "Man Singh Palace", "Bharat Kala Bhavan", "Blue Lassi Shop"]

rest = out[~out["name"].isin(famous)]
quota = {"ghat": 6, "worship": 5, "restaurant": 6, "cafe": 4,
         "park_garden": 3, "historic": 1, "attraction": 1, "museum": 2}
parts = [rest[rest["category"] == c].sample(min(n, (rest["category"] == c).sum()), random_state=42)
         for c, n in quota.items()]
survey = pd.concat([out[out["name"].isin(famous)], *parts]).reset_index(drop=True)
survey.to_csv("../data/processed/survey_candidates.csv", index=False)
print(len(survey))
print(survey[["name", "category"]].to_string())

40
                                  name     category
0         Sankat Mochan Hanuman Temple      worship
1                           Tulsi Ghat         ghat
2                        Ramnagar Fort       museum
3                   Harishchandra Ghat         ghat
4                     Manikarnika Ghat         ghat
5                      Blue Lassi Shop         cafe
6                         Durga Temple      worship
7                   Dashashvamedh Ghat         ghat
8                     Man Singh Palace       museum
9                   Bharat Kala Bhavan       museum
10                  Tulsi Manas Temple      worship
11             Kashi Vishwanath Temple      worship
12                     Manasarowa Ghat         ghat
13         Shri Panch Agni Akhada Ghat         ghat
14                    Anandamayee Ghat         ghat
15                        Hanuman Ghat         ghat
16                      Niranjani Ghat         ghat
17                           Namo Ghat         ghat
18       

In [15]:
rename = {
    "द ग्रीन बाल्कनी कैफे & डिनर": "The Green Balcony Cafe & Dinner",
    "लूसी'स हेरिटेज कैफ़े": "Lucy's Heritage Cafe",
}
survey["name"] = survey["name"].replace(rename)

drop = ["Kundan First Engine of BLW", "foodie friend's", "Tit Bit cafe at home"]
survey = survey[~survey["name"].isin(drop)]

pool = out[~out["name"].isin(set(survey["name"]) | set(drop))]
pool = pool[~pool["name"].str.contains("lucy", case=False)]
extra = pd.concat([pool[pool["category"] == c].sample(1, random_state=7)
                   for c in ["historic", "restaurant", "cafe"]])
print(extra[["name", "category", "lat", "lon"]].to_string())

survey = pd.concat([survey, extra]).reset_index(drop=True)
survey.to_csv("../data/processed/survey_candidates.csv", index=False)
print(len(survey))

                   name    category        lat        lon
251  THE ROAD NOT TAKEN    historic  25.263019  82.984505
308           dosa king  restaurant  25.287799  82.998446
89   Maitri Jalpan Grih        cafe  25.271204  82.993081
40


In [16]:
survey = survey[~survey["name"].isin(["THE ROAD NOT TAKEN", "Maitri Jalpan Grih"])]
used = set(survey["name"])

print("HISTORIC / ATTRACTION candidates:")
print(out[out["category"].isin(["historic", "attraction"]) & ~out["name"].isin(used)][["name", "category", "lat", "lon"]].to_string())

print("\nCAFE candidates:")
print(out[(out["category"] == "cafe") & ~out["name"].isin(used)].sample(12, random_state=3)[["name", "lat", "lon"]].to_string())

HISTORIC / ATTRACTION candidates:
                            name    category        lat        lon
120  Mandakini Meditation Center  attraction  25.263778  83.018599
159         MADAN MOHAN MALAVIYA  attraction  25.275534  83.000831
193                  Lolark Kund  attraction  25.291014  83.005801
198   Kundan First Engine of BLW    historic  25.285356  82.961018
239                     FOUNTAIN  attraction  25.266032  82.988907
251           THE ROAD NOT TAKEN    historic  25.263019  82.984505
253         WAY TO SWIMMING POOL    historic  25.265519  82.986526
262          Giant Praying Hands  attraction  25.325176  83.035846
265             Half Sunk Temple  attraction  25.311480  83.014979
274    Rani Laxmibai Janmasthali  attraction  25.289804  83.005634
304                     Aircraft    historic  25.331446  82.985457
305                     Airplane    historic  25.329701  82.972736
332                  रामनगर किला    historic  25.269707  83.024496

CAFE candidates:
          

In [17]:
survey = survey[~survey["name"].isin(["THE ROAD NOT TAKEN", "Maitri Jalpan Grih", "Shri Daitra veer baba temple"])]
add = out[out["name"].isin(["Half Sunk Temple", "Minnano Cafe"])]
assi = pd.DataFrame([{"name": "Assi Ghat", "category": "ghat", "lat": 25.288, "lon": 83.006, "opening_hours": None}])
survey = pd.concat([survey, add, assi]).reset_index(drop=True)
survey.to_csv("../data/processed/survey_candidates.csv", index=False)
print(len(survey))
print(survey["category"].value_counts())

40
category
ghat           11
worship         8
restaurant      6
museum          5
cafe            5
park_garden     3
attraction      2
Name: count, dtype: int64
